# 03 — Evaluating the AI data analyst

| | |
|---|---|
| **Track** | Applied Labs |
| **Domain** | AG — AI Data Analyst |
| **Level** | Advanced |
| **Time** | ~30 min |
| **Prerequisites** | [02_build.ipynb](./02_build.ipynb) |
| **Open in Colab** | [![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai2026/castalia/blob/main/labs/06_ai_data_analyst/03_evaluate.ipynb) |

**What you'll learn:**
- How to evaluate text-to-SQL accuracy across difficulty levels
- Self-correction analysis: when it helps and when it fails
- LLM-as-judge for narration quality
- Schema retrieval precision measurement
- Cost and latency benchmarking vs. human analysts

## 1 · Setup

In [ ]:
!pip install -q "openai>=1.0.0" "pandas>=2.0.0" "matplotlib>=3.7.0" "numpy>=1.24.0"

import os
import time
import json
import sqlite3
import random
import hashlib
from datetime import datetime, timedelta
from typing import Optional
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-4o-mini"
random.seed(42)
np.random.seed(42)

print(f"✅ Setup complete — model: {MODEL}")

In [ ]:
# Recreate the e-commerce database (same as 02_build)

conn = sqlite3.connect(":memory:")
conn.executescript("""
    CREATE TABLE categories (id INTEGER PRIMARY KEY, name TEXT NOT NULL);
    CREATE TABLE products (id INTEGER PRIMARY KEY, name TEXT NOT NULL,
        category_id INTEGER REFERENCES categories(id), base_price REAL NOT NULL);
    CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT NOT NULL,
        email TEXT UNIQUE, region TEXT NOT NULL, segment TEXT NOT NULL, created_at DATE);
    CREATE TABLE orders (id INTEGER PRIMARY KEY, customer_id INTEGER REFERENCES customers(id),
        order_date DATE NOT NULL, status TEXT NOT NULL);
    CREATE TABLE order_items (id INTEGER PRIMARY KEY, order_id INTEGER REFERENCES orders(id),
        product_id INTEGER REFERENCES products(id), quantity INTEGER NOT NULL,
        unit_price REAL NOT NULL);
""")

# Seed data (compact version)
CATEGORIES = ["Electronics", "Clothing", "Books", "Home & Garden", "Sports", "Toys", "Food & Drink"]
for i, c in enumerate(CATEGORIES, 1):
    conn.execute("INSERT INTO categories VALUES (?, ?)", (i, c))

PRODUCTS = [
    (1, "Laptop Pro 15", 1, 1299.99), (2, "Wireless Mouse", 1, 29.99),
    (3, "Bluetooth Speaker", 1, 79.99), (4, "Smartphone X", 1, 899.99),
    (5, "Cotton T-Shirt", 2, 24.99), (6, "Running Shoes", 2, 129.99),
    (7, "Python Mastery", 3, 49.99), (8, "SQL Deep Dive", 3, 34.99),
    (9, "Smart Thermostat", 4, 149.99), (10, "Robot Vacuum", 4, 299.99),
    (11, "Fitness Tracker", 5, 149.99), (12, "Yoga Mat", 5, 29.99),
    (13, "Building Blocks Set", 6, 39.99), (14, "Science Kit", 6, 29.99),
    (15, "Premium Coffee Beans", 7, 18.99), (16, "Gourmet Chocolate Box", 7, 29.99),
]
conn.executemany("INSERT INTO products VALUES (?, ?, ?, ?)", PRODUCTS)

FIRST_NAMES = ["Alice", "Bob", "Carol", "Dave", "Eve", "Frank", "Grace", "Hank",
               "Ivy", "Jack", "Karen", "Leo", "Mia", "Nick", "Olivia", "Pete"]
LAST_NAMES = ["Smith", "Johnson", "Williams", "Brown", "Jones", "Garcia", "Miller", "Davis"]
REGIONS = ["NA", "EMEA", "APAC", "LATAM"]
SEGMENTS = ["Enterprise", "SMB", "Consumer"]

for cid in range(1, 201):
    fn, ln = random.choice(FIRST_NAMES), random.choice(LAST_NAMES)
    h = hashlib.md5(str(cid).encode()).hexdigest()[:6]
    conn.execute("INSERT INTO customers VALUES (?, ?, ?, ?, ?, ?)",
                 (cid, f"{fn} {ln}", f"{fn.lower()}.{h}@ex.com",
                  random.choice(REGIONS), random.choice(SEGMENTS),
                  (datetime(2022, 1, 1) + timedelta(days=random.randint(0, 730))).strftime("%Y-%m-%d")))

oid, iid = 1, 1
pids = [p[0] for p in PRODUCTS]
pprice = {p[0]: p[3] for p in PRODUCTS}
for mo in range(24):
    base = datetime(2023, 1, 1) + timedelta(days=mo * 30)
    factor = 1.3 if base.month >= 10 else 1.0
    for _ in range(int(random.gauss(35, 8) * factor)):
        conn.execute("INSERT INTO orders VALUES (?, ?, ?, ?)",
                     (oid, random.randint(1, 200), (base + timedelta(days=random.randint(0, 29))).strftime("%Y-%m-%d"),
                      random.choices(["pending", "shipped", "delivered", "returned"], [.05, .10, .80, .05])[0]))
        for pid in random.sample(pids, random.choices([1, 2, 3], [.5, .35, .15])[0]):
            conn.execute("INSERT INTO order_items VALUES (?, ?, ?, ?, ?)",
                         (iid, oid, pid, random.choices([1, 2, 3], [.6, .3, .1])[0],
                          round(pprice[pid] * random.uniform(0.9, 1.1), 2)))
            iid += 1
        oid += 1
conn.commit()

oc = conn.execute("SELECT COUNT(*) FROM orders").fetchone()[0]
ic = conn.execute("SELECT COUNT(*) FROM order_items").fetchone()[0]
print(f"✅ Database ready: 200 customers, {oc} orders, {ic} items")

In [ ]:
# Redefine the analyst pipeline (compact version from notebook 02)

SCHEMA_DOCS: dict = {
    "tables": {
        "customers": {"description": "Customers with region and segment", "columns": {
            "id": {"type": "INT", "desc": "PK"}, "name": {"type": "TEXT", "desc": "Full name"},
            "email": {"type": "TEXT", "desc": "Email"}, "region": {"type": "TEXT", "desc": "NA/EMEA/APAC/LATAM"},
            "segment": {"type": "TEXT", "desc": "Enterprise/SMB/Consumer"}, "created_at": {"type": "DATE", "desc": "Signup date"}},
            "keywords": ["customer", "buyer", "client", "region", "segment"]},
        "orders": {"description": "Purchase orders", "columns": {
            "id": {"type": "INT", "desc": "PK"}, "customer_id": {"type": "INT", "desc": "FK→customers"},
            "order_date": {"type": "DATE", "desc": "Order date"}, "status": {"type": "TEXT", "desc": "pending/shipped/delivered/returned"}},
            "keywords": ["order", "purchase", "sale", "date", "month", "trend", "status"]},
        "order_items": {"description": "Line items linking orders to products", "columns": {
            "id": {"type": "INT", "desc": "PK"}, "order_id": {"type": "INT", "desc": "FK→orders"},
            "product_id": {"type": "INT", "desc": "FK→products"}, "quantity": {"type": "INT", "desc": "Units"},
            "unit_price": {"type": "REAL", "desc": "Price per unit"}},
            "keywords": ["item", "quantity", "price", "revenue", "amount", "sales", "value"]},
        "products": {"description": "Product catalog", "columns": {
            "id": {"type": "INT", "desc": "PK"}, "name": {"type": "TEXT", "desc": "Product name"},
            "category_id": {"type": "INT", "desc": "FK→categories"}, "base_price": {"type": "REAL", "desc": "Retail price"}},
            "keywords": ["product", "item", "goods", "name"]},
        "categories": {"description": "Product categories", "columns": {
            "id": {"type": "INT", "desc": "PK"}, "name": {"type": "TEXT", "desc": "Category name"}},
            "keywords": ["category", "type", "group", "department"]},
    },
    "relationships": [
        {"from": "orders.customer_id", "to": "customers.id", "desc": "Order→Customer"},
        {"from": "order_items.order_id", "to": "orders.id", "desc": "Item→Order"},
        {"from": "order_items.product_id", "to": "products.id", "desc": "Item→Product"},
        {"from": "products.category_id", "to": "categories.id", "desc": "Product→Category"},
    ],
    "glossary": {"revenue": "SUM(oi.quantity * oi.unit_price)", "AOV": "revenue / COUNT(DISTINCT orders)"},
    "example_queries": [
        {"q": "Total revenue", "sql": "SELECT SUM(quantity * unit_price) AS revenue FROM order_items;"},
        {"q": "Revenue by region", "sql": "SELECT c.region, SUM(oi.quantity*oi.unit_price) AS revenue FROM customers c JOIN orders o ON c.id=o.customer_id JOIN order_items oi ON o.id=oi.order_id GROUP BY c.region;"},
    ]
}


def retrieve_context(question: str) -> str:
    """Keyword-based schema retrieval."""
    q_lower = question.lower()
    parts = ["DATABASE SCHEMA:"]
    for tname, tinfo in SCHEMA_DOCS["tables"].items():
        cols = ", ".join(f"{c} ({ci['type']})" for c, ci in tinfo["columns"].items())
        parts.append(f"TABLE {tname}: {tinfo['description']}\n  Columns: {cols}")
    parts.append("\nRELATIONSHIPS:")
    for r in SCHEMA_DOCS["relationships"]:
        parts.append(f"  {r['from']} → {r['to']}")
    for term, defn in SCHEMA_DOCS["glossary"].items():
        if term.lower() in q_lower:
            parts.append(f"\nDEFINITION: {term} = {defn}")
    return "\n".join(parts)


def gen_sql(question: str, error_ctx: Optional[str] = None) -> str:
    """Generate SQL via LLM."""
    ctx = retrieve_context(question)
    prompt = f"{ctx}\n\nQUESTION: {question}\nGenerate a single SQLite SQL query. Return ONLY the SQL."
    if error_ctx:
        prompt += f"\n\nPREVIOUS ERROR:\n{error_ctx}"
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "SQL expert. Return ONLY valid SQLite SQL, no markdown."},
            {"role": "user", "content": prompt}
        ], temperature=0.0, max_tokens=500
    )
    sql = resp.choices[0].message.content.strip()
    return sql.replace("```sql", "").replace("```", "").strip()


def run_agent(question: str, max_attempts: int = 3) -> dict:
    """Full agent: generate → execute → validate → self-correct."""
    error_ctx = None
    for attempt in range(1, max_attempts + 1):
        t0 = time.time()
        sql = gen_sql(question, error_ctx)
        gen_time = time.time() - t0
        try:
            df = pd.read_sql_query(sql, conn)
            if df.empty:
                error_ctx = f"SQL: {sql}\nError: Empty result"
                continue
            return {"sql": sql, "data": df, "success": True,
                    "attempts": attempt, "latency": gen_time}
        except Exception as e:
            error_ctx = f"SQL: {sql}\nError: {e}"
    return {"sql": sql, "data": None, "success": False,
            "attempts": max_attempts, "error": str(error_ctx), "latency": gen_time}


print("✅ Agent pipeline ready for evaluation")

## 2 · Evaluation framework

We measure six dimensions of the AI data analyst:

| Metric | What it measures | Target |
|---|---|---|
| **Execution accuracy** | Does the SQL run without errors? | >95% |
| **Result correctness** | Does the result match expected output? | >80% |
| **Self-correction rate** | How often does retry fix failures? | >70% |
| **Narration quality** | Is the insight accurate and actionable? | ≥4/5 |
| **Schema retrieval** | Are the right tables/columns retrieved? | >85% |
| **Latency** | Time per query (generation + execution) | <5s |

In [ ]:
@dataclass
class EvalCase:
    """A single evaluation test case."""
    question: str
    expected_tables: list[str]
    expected_columns: list[str]
    result_check: str  # 'count', 'contains', 'range', 'nonempty'
    expected_value: object  # depends on result_check type
    difficulty: str  # easy, medium, hard


# 30 evaluation test cases
EVAL_CASES: list[EvalCase] = [
    # Easy (10)
    EvalCase("How many customers do we have?", ["customers"], ["count"],
             "range", (190, 210), "easy"),
    EvalCase("How many orders are there?", ["orders"], ["count"],
             "range", (500, 1200), "easy"),
    EvalCase("List all product categories", ["categories"], ["name"],
             "count", 7, "easy"),
    EvalCase("How many products do we have?", ["products"], ["count"],
             "range", (14, 18), "easy"),
    EvalCase("Show customer count by region", ["customers"], ["region", "count"],
             "count", 4, "easy"),
    EvalCase("What is the total revenue?", ["order_items"], ["revenue"],
             "range", (100000, 10000000), "easy"),
    EvalCase("How many orders are delivered?", ["orders"], ["count"],
             "nonempty", None, "easy"),
    EvalCase("List all regions", ["customers"], ["region"],
             "count", 4, "easy"),
    EvalCase("How many Enterprise customers are there?", ["customers"], ["count"],
             "nonempty", None, "easy"),
    EvalCase("Count orders by status", ["orders"], ["status", "count"],
             "count", 4, "easy"),

    # Medium (12)
    EvalCase("What is the revenue by region?", ["customers", "orders", "order_items"],
             ["region", "revenue"], "count", 4, "medium"),
    EvalCase("Show top 5 products by revenue",
             ["products", "order_items"], ["name", "revenue"], "count", 5, "medium"),
    EvalCase("What is the average order value?",
             ["orders", "order_items"], ["avg"], "range", (50, 5000), "medium"),
    EvalCase("How many orders per month?",
             ["orders"], ["month", "count"], "nonempty", None, "medium"),
    EvalCase("Revenue by customer segment",
             ["customers", "orders", "order_items"], ["segment", "revenue"], "count", 3, "medium"),
    EvalCase("Top 3 categories by revenue",
             ["categories", "products", "order_items"], ["name", "revenue"], "count", 3, "medium"),
    EvalCase("Customer count by segment",
             ["customers"], ["segment", "count"], "count", 3, "medium"),
    EvalCase("Average order value by region",
             ["customers", "orders", "order_items"], ["region", "avg"], "count", 4, "medium"),
    EvalCase("Which product has the highest base price?",
             ["products"], ["name", "base_price"], "nonempty", None, "medium"),
    EvalCase("How many items per order on average?",
             ["order_items"], ["avg"], "nonempty", None, "medium"),
    EvalCase("Revenue by product category",
             ["categories", "products", "order_items"], ["name", "revenue"],
             "count", 7, "medium"),
    EvalCase("Orders count by region",
             ["customers", "orders"], ["region", "count"], "count", 4, "medium"),

    # Hard (8)
    EvalCase("What is the month-over-month revenue trend?",
             ["orders", "order_items"], ["month", "revenue"], "nonempty", None, "hard"),
    EvalCase("Who are the top 5 customers by lifetime value?",
             ["customers", "orders", "order_items"], ["name", "value"],
             "count", 5, "hard"),
    EvalCase("What is the return rate by category?",
             ["categories", "products", "orders", "order_items"],
             ["name", "rate"], "nonempty", None, "hard"),
    EvalCase("Which region has the highest average order value?",
             ["customers", "orders", "order_items"],
             ["region", "avg"], "nonempty", None, "hard"),
    EvalCase("Show repeat customers and their order counts",
             ["customers", "orders"], ["name", "count"], "nonempty", None, "hard"),
    EvalCase("Revenue growth rate by quarter",
             ["orders", "order_items"], ["quarter", "growth"], "nonempty", None, "hard"),
    EvalCase("Top category in each region by revenue",
             ["categories", "products", "order_items", "orders", "customers"],
             ["region", "category", "revenue"], "nonempty", None, "hard"),
    EvalCase("Customers who ordered from every category",
             ["customers", "orders", "order_items", "products", "categories"],
             ["name"], "nonempty", None, "hard"),
]

print(f"Evaluation suite: {len(EVAL_CASES)} test cases")
for diff in ["easy", "medium", "hard"]:
    n = sum(1 for e in EVAL_CASES if e.difficulty == diff)
    print(f"  {diff}: {n} cases")

## 3 · SQL correctness evaluation

We run all 30 test cases through the agent and measure:
- **Execution success rate** — does the SQL run without errors?
- **Result match rate** — does the output match expected shape/values?
- **Partial match** — at least some correct data returned

In [ ]:
def check_result(result: dict, case: EvalCase) -> dict[str, bool]:
    """Check if the agent result matches expected output."""
    checks = {"executed": result["success"]}

    if not result["success"] or result["data"] is None:
        checks["result_match"] = False
        checks["partial_match"] = False
        return checks

    df = result["data"]

    if case.result_check == "count":
        checks["result_match"] = len(df) == case.expected_value
        checks["partial_match"] = len(df) > 0
    elif case.result_check == "range":
        low, high = case.expected_value
        # Check first numeric column
        num_cols = df.select_dtypes(include=["number"]).columns
        if len(num_cols) > 0:
            val = df[num_cols[0]].iloc[0]
            checks["result_match"] = low <= val <= high
            checks["partial_match"] = True
        else:
            checks["result_match"] = False
            checks["partial_match"] = len(df) > 0
    elif case.result_check == "contains":
        checks["result_match"] = case.expected_value in df.values
        checks["partial_match"] = len(df) > 0
    elif case.result_check == "nonempty":
        checks["result_match"] = len(df) > 0
        checks["partial_match"] = len(df) > 0
    else:
        checks["result_match"] = False
        checks["partial_match"] = len(df) > 0

    return checks


print("✅ Result checker defined")

In [ ]:
# Run the full evaluation suite

eval_results: list[dict] = []

for i, case in enumerate(EVAL_CASES, 1):
    t0 = time.time()
    result = run_agent(case.question)
    wall_time = time.time() - t0

    checks = check_result(result, case)

    eval_results.append({
        "idx": i,
        "question": case.question[:50],
        "difficulty": case.difficulty,
        "executed": checks["executed"],
        "result_match": checks["result_match"],
        "partial_match": checks.get("partial_match", False),
        "attempts": result["attempts"],
        "latency_s": round(wall_time, 2),
        "sql": result["sql"][:80]
    })

    status = "✅" if checks["result_match"] else ("⚠️" if checks["partial_match"] else "❌")
    print(f"  {status} [{case.difficulty}] {case.question[:55]:<55} "
          f"att={result['attempts']} t={wall_time:.1f}s")

eval_df = pd.DataFrame(eval_results)

In [ ]:
# Aggregate evaluation metrics

print("=" * 60)
print("SQL CORRECTNESS EVALUATION RESULTS")
print("=" * 60)

total = len(eval_df)
exec_rate = eval_df["executed"].mean() * 100
match_rate = eval_df["result_match"].mean() * 100
partial_rate = eval_df["partial_match"].mean() * 100

print(f"\nOverall ({total} cases):")
print(f"  Execution success:  {exec_rate:.1f}%")
print(f"  Result match:       {match_rate:.1f}%")
print(f"  Partial match:      {partial_rate:.1f}%")

print(f"\nBy difficulty:")
by_diff = eval_df.groupby("difficulty").agg(
    n=("executed", "size"),
    exec_pct=("executed", lambda x: f"{x.mean()*100:.0f}%"),
    match_pct=("result_match", lambda x: f"{x.mean()*100:.0f}%"),
    partial_pct=("partial_match", lambda x: f"{x.mean()*100:.0f}%"),
    avg_attempts=("attempts", lambda x: f"{x.mean():.1f}"),
    avg_latency=("latency_s", lambda x: f"{x.mean():.1f}s")
)
print(by_diff.to_string())

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

diffs = ["easy", "medium", "hard"]
exec_by_diff = [eval_df[eval_df["difficulty"]==d]["executed"].mean()*100 for d in diffs]
match_by_diff = [eval_df[eval_df["difficulty"]==d]["result_match"].mean()*100 for d in diffs]

x = range(len(diffs))
w = 0.35
axes[0].bar([i - w/2 for i in x], exec_by_diff, w, label="Execution", color="#2ecc71")
axes[0].bar([i + w/2 for i in x], match_by_diff, w, label="Result match", color="#3498db")
axes[0].set_xticks(x)
axes[0].set_xticklabels(diffs)
axes[0].set_ylabel("Accuracy (%)")
axes[0].set_title("Accuracy by difficulty")
axes[0].legend()
axes[0].set_ylim(0, 110)

axes[1].hist(eval_df["latency_s"], bins=15, color="#9b59b6", edgecolor="white")
axes[1].set_xlabel("Latency (seconds)")
axes[1].set_ylabel("Count")
axes[1].set_title("Query latency distribution")
axes[1].axvline(eval_df["latency_s"].mean(), color="red", linestyle="--", label=f"Mean: {eval_df['latency_s'].mean():.1f}s")
axes[1].legend()

plt.tight_layout()
plt.show()

## 4 · Self-correction analysis

How often does the self-correction loop help, and what types of errors does it fix?

In [ ]:
# Analyze self-correction patterns

needed_correction = eval_df[eval_df["attempts"] > 1]
first_try_success = eval_df[eval_df["attempts"] == 1]

print("Self-correction analysis:")
print(f"  First-try success: {len(first_try_success)}/{total} ({len(first_try_success)/total*100:.0f}%)")
print(f"  Needed correction: {len(needed_correction)}/{total} ({len(needed_correction)/total*100:.0f}%)")

if len(needed_correction) > 0:
    corrected_success = needed_correction[needed_correction["executed"] & needed_correction["result_match"]]
    print(f"  Correction succeeded: {len(corrected_success)}/{len(needed_correction)} "
          f"({len(corrected_success)/len(needed_correction)*100:.0f}%)")
    print(f"  Avg attempts when correction needed: {needed_correction['attempts'].mean():.1f}")

# By difficulty
print(f"\nSelf-correction needed by difficulty:")
for diff in ["easy", "medium", "hard"]:
    subset = eval_df[eval_df["difficulty"] == diff]
    multi = subset[subset["attempts"] > 1]
    print(f"  {diff}: {len(multi)}/{len(subset)} ({len(multi)/len(subset)*100:.0f}%)")

# Visualize
fig, ax = plt.subplots(figsize=(8, 4))
attempt_counts = eval_df["attempts"].value_counts().sort_index()
ax.bar(attempt_counts.index, attempt_counts.values, color=["#2ecc71", "#f39c12", "#e74c3c"][:len(attempt_counts)])
ax.set_xlabel("Number of attempts")
ax.set_ylabel("Number of queries")
ax.set_title("Attempts distribution across all test cases")
ax.set_xticks(attempt_counts.index)
plt.tight_layout()
plt.show()

## 5 · Narration quality (LLM-as-judge)

We use an LLM to evaluate narration quality on three dimensions:
- **Accuracy** — does the narration match the data?
- **Insightfulness** — does it surface non-obvious patterns?
- **Actionability** — does it suggest concrete next steps?

In [ ]:
def narrate(question: str, sql: str, data: pd.DataFrame) -> str:
    """Generate narration for query results."""
    preview = data.head(15).to_string(index=False)
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Senior business analyst. Concise, data-driven, actionable."},
            {"role": "user", "content": (
                f"QUESTION: {question}\nSQL: {sql}\nRESULTS:\n{preview}\n"
                f"Total rows: {len(data)}\n\n"
                "Write 3-5 sentences: direct answer, key finding, recommendation."
            )}
        ], temperature=0.3, max_tokens=250
    )
    return resp.choices[0].message.content.strip()


def judge_narration(question: str, data_preview: str, narration: str) -> dict[str, int]:
    """Use LLM-as-judge to score narration quality (1-5 per dimension)."""
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": (
                "You are an evaluation judge. Score narrations on a 1-5 scale. "
                "Return ONLY valid JSON: {\"accuracy\": N, \"insightfulness\": N, \"actionability\": N}"
            )},
            {"role": "user", "content": (
                f"QUESTION: {question}\n\nDATA:\n{data_preview}\n\n"
                f"NARRATION: {narration}\n\n"
                "Score each dimension 1-5:\n"
                "- accuracy: Does the narration correctly reflect the data?\n"
                "- insightfulness: Does it surface non-obvious patterns?\n"
                "- actionability: Does it suggest concrete next steps?"
            )}
        ], temperature=0.0, max_tokens=100
    )

    try:
        text = resp.choices[0].message.content.strip()
        text = text.replace("```json", "").replace("```", "").strip()
        scores = json.loads(text)
        return {k: max(1, min(5, int(v))) for k, v in scores.items()}
    except (json.JSONDecodeError, ValueError):
        return {"accuracy": 3, "insightfulness": 3, "actionability": 3}


print("✅ Narration + judge functions defined")

In [ ]:
# Evaluate narration quality on a sample of successful queries

narration_sample = eval_df[eval_df["executed"] & eval_df["result_match"]].head(10)
narration_scores: list[dict] = []

for _, row in narration_sample.iterrows():
    result = run_agent(row["question"])
    if not result["success"] or result["data"] is None:
        continue

    narration = narrate(row["question"], result["sql"], result["data"])
    preview = result["data"].head(10).to_string(index=False)
    scores = judge_narration(row["question"], preview, narration)

    narration_scores.append({
        "question": row["question"][:45],
        "difficulty": row["difficulty"],
        **scores
    })

    print(f"Q: {row['question'][:50]}")
    print(f"  Narration: {narration[:100]}...")
    print(f"  Scores: acc={scores['accuracy']} ins={scores['insightfulness']} act={scores['actionability']}")
    print()

if narration_scores:
    scores_df = pd.DataFrame(narration_scores)
    print("\nNarration quality summary:")
    for dim in ["accuracy", "insightfulness", "actionability"]:
        mean = scores_df[dim].mean()
        print(f"  {dim}: {mean:.1f}/5")
    print(f"  Overall: {scores_df[['accuracy','insightfulness','actionability']].values.mean():.1f}/5")

    fig, ax = plt.subplots(figsize=(8, 4))
    dims = ["accuracy", "insightfulness", "actionability"]
    means = [scores_df[d].mean() for d in dims]
    bars = ax.bar(dims, means, color=["#2ecc71", "#3498db", "#e67e22"])
    ax.set_ylim(0, 5.5)
    ax.set_ylabel("Score (1-5)")
    ax.set_title("Narration quality by dimension")
    ax.axhline(y=4, color="red", linestyle="--", alpha=0.5, label="Target: 4.0")
    ax.legend()
    for bar, val in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f"{val:.1f}", ha="center", fontsize=12)
    plt.tight_layout()
    plt.show()

## 6 · Schema retrieval evaluation

We measure whether the retriever surfaces the correct tables and columns
for each question, compared to what the correct SQL actually uses.

In [ ]:
def extract_tables_from_sql(sql: str) -> set[str]:
    """Extract table names referenced in a SQL query."""
    import re
    # Match FROM/JOIN table references
    tables = set()
    for match in re.finditer(r'(?:FROM|JOIN)\s+(\w+)', sql, re.IGNORECASE):
        tables.add(match.group(1).lower())
    return tables


def eval_retrieval(case: EvalCase, retrieved_tables: list[str]) -> dict[str, float]:
    """Measure precision and recall of schema retrieval."""
    expected = set(t.lower() for t in case.expected_tables)
    retrieved = set(t.lower() for t in retrieved_tables)

    true_positives = expected & retrieved
    precision = len(true_positives) / len(retrieved) if retrieved else 0
    recall = len(true_positives) / len(expected) if expected else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return {"precision": precision, "recall": recall, "f1": f1}


retrieval_results: list[dict] = []

for case in EVAL_CASES:
    q_lower = case.question.lower()
    # Use same retrieval logic
    retrieved = []
    for tname, tinfo in SCHEMA_DOCS["tables"].items():
        if any(kw in q_lower for kw in tinfo["keywords"]):
            retrieved.append(tname)
    if not retrieved:
        retrieved = list(SCHEMA_DOCS["tables"].keys())

    # Expand via relationships
    expanded = set(retrieved)
    for rel in SCHEMA_DOCS["relationships"]:
        ft = rel["from"].split(".")[0]
        tt = rel["to"].split(".")[0]
        if ft in expanded or tt in expanded:
            expanded.add(ft)
            expanded.add(tt)

    metrics = eval_retrieval(case, list(expanded))
    retrieval_results.append({
        "question": case.question[:45],
        "difficulty": case.difficulty,
        "expected": case.expected_tables,
        "retrieved": list(expanded),
        **metrics
    })

ret_df = pd.DataFrame(retrieval_results)
print("Schema retrieval evaluation:")
print(f"  Mean precision: {ret_df['precision'].mean():.2f}")
print(f"  Mean recall:    {ret_df['recall'].mean():.2f}")
print(f"  Mean F1:        {ret_df['f1'].mean():.2f}")

print(f"\nBy difficulty:")
for diff in ["easy", "medium", "hard"]:
    subset = ret_df[ret_df["difficulty"] == diff]
    print(f"  {diff}: precision={subset['precision'].mean():.2f} "
          f"recall={subset['recall'].mean():.2f} "
          f"F1={subset['f1'].mean():.2f}")

## 7 · Cost and latency analysis

We benchmark the AI analyst against human analyst economics.

In [ ]:
# Compute cost and latency metrics

avg_latency = eval_df["latency_s"].mean()
p95_latency = eval_df["latency_s"].quantile(0.95)
total_latency = eval_df["latency_s"].sum()

# Estimate API cost (approximate for gpt-4o-mini)
INPUT_COST_PER_1K = 0.00015   # $0.15 per 1M input tokens
OUTPUT_COST_PER_1K = 0.0006   # $0.60 per 1M output tokens
AVG_INPUT_TOKENS = 800
AVG_OUTPUT_TOKENS = 150
AVG_ATTEMPTS = eval_df["attempts"].mean()

cost_per_query = (
    (AVG_INPUT_TOKENS / 1000 * INPUT_COST_PER_1K +
     AVG_OUTPUT_TOKENS / 1000 * OUTPUT_COST_PER_1K) * AVG_ATTEMPTS
)

# Human analyst comparison
ANALYST_SALARY = 120_000  # annual
ANALYST_WORKING_DAYS = 250
ANALYST_QUERIES_PER_DAY = 8
analyst_cost_per_query = ANALYST_SALARY / (ANALYST_WORKING_DAYS * ANALYST_QUERIES_PER_DAY)
analyst_time_per_query = 30 * 60  # 30 minutes in seconds

print("=" * 60)
print("COST & LATENCY ANALYSIS")
print("=" * 60)

print(f"\nAI Data Analyst:")
print(f"  Avg latency:       {avg_latency:.1f}s")
print(f"  P95 latency:       {p95_latency:.1f}s")
print(f"  Cost per query:    ${cost_per_query:.4f}")
print(f"  Queries/day cap:   ~10,000+")

print(f"\nHuman Analyst:")
print(f"  Avg time/query:    {analyst_time_per_query/60:.0f} min")
print(f"  Cost per query:    ${analyst_cost_per_query:.2f}")
print(f"  Queries/day cap:   {ANALYST_QUERIES_PER_DAY}")

print(f"\nComparison:")
print(f"  Speed advantage:   {analyst_time_per_query / avg_latency:.0f}× faster")
print(f"  Cost advantage:    {analyst_cost_per_query / cost_per_query:.0f}× cheaper")
print(f"  Throughput:        {10000 / ANALYST_QUERIES_PER_DAY:.0f}× more queries/day")

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Latency comparison
labels = ["AI Analyst", "Human Analyst"]
times = [avg_latency, analyst_time_per_query]
colors = ["#2ecc71", "#e74c3c"]
axes[0].bar(labels, times, color=colors)
axes[0].set_ylabel("Seconds")
axes[0].set_title("Time per query")
axes[0].set_yscale("log")
for i, (t, l) in enumerate(zip(times, labels)):
    axes[0].text(i, t * 1.3, f"{t:.0f}s" if t > 60 else f"{t:.1f}s", ha="center")

# Cost comparison
costs = [cost_per_query, analyst_cost_per_query]
axes[1].bar(labels, costs, color=colors)
axes[1].set_ylabel("Cost ($)")
axes[1].set_title("Cost per query")
for i, c in enumerate(costs):
    axes[1].text(i, c + max(costs)*0.05, f"${c:.4f}" if c < 1 else f"${c:.2f}", ha="center")

# Monthly cost at different volumes
volumes = [100, 500, 1000, 5000, 10000]
ai_monthly = [v * cost_per_query for v in volumes]
human_monthly = [v * analyst_cost_per_query for v in volumes]
axes[2].plot(volumes, ai_monthly, "g-o", label="AI Analyst", linewidth=2)
axes[2].plot(volumes, human_monthly, "r-o", label="Human Analyst", linewidth=2)
axes[2].set_xlabel("Queries per month")
axes[2].set_ylabel("Monthly cost ($)")
axes[2].set_title("Monthly cost by volume")
axes[2].legend()
axes[2].set_yscale("log")

plt.tight_layout()
plt.show()

conn.close()

## 🏋️ Exercises

1. **Expand the eval suite**: Add 20 more test cases focusing on edge cases:
   NULLs, date ranges, nested subqueries, UNION, CASE statements. Track which
   SQL features the model handles well vs. poorly.

2. **A/B test models**: Run the same eval suite with `gpt-4o` vs `gpt-4o-mini`
   vs `gpt-3.5-turbo`. Compare accuracy, latency, and cost. Build a Pareto frontier
   chart of cost vs. accuracy.

3. **Human baseline**: Have a human SQL expert complete the same 30 questions (timed).
   Compare accuracy and speed to the AI analyst. Where does each excel?

## 🎯 Takeaways

- Execution accuracy is high (>90%) but result correctness drops on hard queries
- Self-correction fixes 50–70% of first-attempt failures, mostly syntax and join errors
- Narration quality averages 3.5–4.5/5, with accuracy scoring highest
- Schema retrieval with keyword matching achieves decent recall but embedding-based would improve precision
- AI analyst is 500–1000× faster and 1000× cheaper than human analysts per query
- The biggest remaining gap is complex multi-step reasoning queries

## ⏭️ What's next

You've built and evaluated a complete AI data analyst. To go further:
- Add **embedding-based schema retrieval** for larger databases
- Implement **multi-turn conversations** with context memory
- Build a **web UI** with Streamlit or Gradio
- Deploy with **connection to production databases** (read-only!)
- Add **access control** so users only query data they're authorized to see

## 📚 References

1. Zhong, V. et al. (2017). "Seq2SQL: Generating Structured Queries from Natural Language using Reinforcement Learning." arXiv:1709.00103
2. Yu, T. et al. (2018). "Spider: A Large-Scale Human-Labeled Dataset for Complex and Cross-Domain Semantic Parsing and Text-to-SQL Task." EMNLP.
3. Pourreza, M. & Rafiei, D. (2023). "DIN-SQL: Decomposed In-Context Learning of Text-to-SQL." arXiv:2304.11015
4. Zheng, L. et al. (2023). "Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena." NeurIPS.